In [ ]:
# 허깅페이스 - 가장 핵심적인 세개 라이브러리
# transformers : bert, gpt 등 사전학습된 모델
!pip install -U transformers
# 모델이 학습한 방식대로 텍스트를 쪼갠다(토크나이저)
!pip install -U tokenizers
!pip install -U datasets

In [ ]:
!pip uninstall -y transformers tokenizers

In [ ]:
!pip install transformers

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import os
import shutil
import torch.optim as optim
from transformers import AutoTokenizer

In [ ]:
import urllib.request

def get_file(filename, url):
  # 전체 경로 만듬
  save_file=os.path.join(os.getcwd(), filename)

  if not os.path.exists(save_file):
    print("download")
    # url내용 절대경로+파일명합친 경로에 저장하겠다.
    urllib.request.urlretrieve(url, save_file)
  else:
    print("already exists")

  return save_file

In [ ]:
data_train=get_file("review_train2.txt","https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt" )
data_test=get_file("review_test2.txt","https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt" )

In [ ]:
# read_csv로 데이터 가져와 shape확인한다
train_data=pd.read_csv(data_train, sep='\t')
test_data=pd.read_csv(data_test, sep='\t')

In [ ]:
print(train_data.shape, test_data.shape)

In [ ]:
# 학습 15000, 테스트 5000개로 샘플링(섞는다)
# 결측치 확인
train_data=train_data.sample(n=15000, random_state=1)
test_data=test_data.sample(n=5000, random_state=1)

In [ ]:
print(train_data.isna().sum())
print(test_data.isna().sum())

In [ ]:
train_data.dropna(inplace=True)
print(train_data.isna().sum())

In [ ]:
train_data.head()

In [ ]:
print(train_data.shape, test_data.shape)

In [ ]:
class_name={1:'긍정', 0:'부정'}

In [ ]:
# 커스텀 데이터셋 정의
class MovieReview(Dataset):
  def __init__(self, documents, labels):
    self.documents=documents.tolist()
    self.labels=labels.tolist()

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.documents[idx], self.labels[idx]

In [ ]:
# 데이터셋 객체생성
train_dataset=MovieReview(train_data['document'], train_data['label'])
test_dataset=MovieReview(test_data['document'], test_data['label'])

In [ ]:
train_loader=DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
test_loader=DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
len(train_loader)

In [ ]:
data_iterate=iter(train_loader)
# 첫번째 배치 꺼내옴(32개)
next(data_iterate)

In [ ]:
document_batch, label_batch=next(data_iterate)
document_batch[0]

In [ ]:
class_names=['부정', '긍정']

In [ ]:
for i in range(10):
  text=document_batch[i]
  label=label_batch[i].item()
  print(f'리뷰:{text}')
  print(f'label:{label}({class_name[label]})')

In [ ]:
bert_name='bert-base-multilingual-cased'

model_id=bert_name

In [ ]:
from transformers import AutoTokenizer, AutoModel

# model_id(bert모델명)
# 영화 너무 재미있어요 -> 모델마다 글자를 숫자로 바꾸는 사전, 규칙이 다 다르다
tokenizer=AutoTokenizer.from_pretrained(model_id)
# 구글에서 미리 텍스트로 공부시켜 놓은 사전학습모델
# 이미 학습이 끝난 가중치 다운로드
model=AutoModel.from_pretrained(model_id)

In [ ]:
print(tokenizer)

In [ ]:
print(model)

In [ ]:
text_test=['이 영화 너무 재밌어요!', '내용이 너무 싫다']
text_preprocess=tokenizer(text_test,
                          padding=True,
                          truncation=True,
                          return_tensors='pt')

In [ ]:
print(text_preprocess.keys())
# cls : 101번
# sep : 102번

In [ ]:
print(text_preprocess['input_ids'])

In [ ]:
# bert모델 통과 -> 복합객체
outputs=model(**text_preprocess)
outputs.keys()

In [ ]:
# last_hidden_state : bert 마지막 층에서 나온 모든 토큰 벡터값(문장 내 각 단어 의미파악)
# pooler_output : 문장 전체의 의미를 대표하는 하나 벡터

In [ ]:
# 모델 추론
with torch.no_grad():
  bert_results=model(**text_preprocess)

# 모델 통과 후 [배치크기, 벡터차원]
# 768개의 숫자를 입력받아 최종 판단을 내리게 된다
print(bert_results.pooler_output.shape)

# 문장에 포함된 8개 토큰에 대한 각각의 의미를 가지고 있는 벡터값
# 모델 통과 후 [배치크기, 문장 길이, 벡터 차원]
print(bert_results.last_hidden_state.shape)

In [ ]:
# bert모델 -> 12개 layer

In [ ]:
class BertModel(nn.Module):
  def __init__(self, model_id):
    super().__init__()
    # 언어 문맥 파악하는 가중치와 bert모델 구조
    self.bert=AutoModel.from_pretrained(model_id)
    # 과적합 방지위해 10% 일부 노드 무작위로 끔(학습)
    self.dropout=nn.Dropout(0.1)
    # (768,1) -> linear : bert 뽑아진 특징들 (768)사이에서 14ㅐ
    self.classfier=nn.Linear(self.bert.config.hidden_size,1)

  def forward(self, input_ids, attention_mask, token_type_ids):
    outputs=self.bert(
        input_ids=input_ids,
        attention_mask=attention_mask,
        token_type_ids=token_type_ids)

    pooled_output=outputs.pooler_output

    net=self.dropout(pooled_output)
    logits=self.classfier(net)

    return logits

In [ ]:
# 데이터 들어왔을때 흐름
# 1. bert 통과 : input_ids, ..... 받아 문맥 파악
# 2. 문장 대표하는 768차원 뽑아옴
# 3. 뽑은 벡터에 dropout 입힘
# 4. 마지막 선형 레이어를 통과시켜 최종 점수를 만듬

In [ ]:
model_id='bert-base-multilingual-cased'
model=BertModel(model_id)

In [ ]:
# dropout이 0%로 고정 -> 모든 노드 활성화
model.eval()

inputs={k:v.to(model.bert.device) for k,v in text_preprocess.items()}

with torch.no_grad():
  bert_results=model(**inputs)

print(bert_results)

prob=torch.sigmoid(bert_results)
print(prob)

In [ ]:
!pip install torchinfo

In [ ]:
# 요약표 제공해주는 라이브러리
from torchinfo import summary

In [ ]:
# forward 함수 인자가 3개(input_ids(배치사이즈, 문장길이), mask, type_ids)
# 가상데이터를 넣어 모델 한번 돌려보려고 -> 흐름 추적
summary(model, input_size=[(1,12),(1,12),(1,12)],
        dtypes=[torch.long, torch.long, torch.long],
        col_names=["input_size", "output_size", "num_params", "kernel_size"],
        depth=3)

In [ ]:
def binary_accuracy(preds, y):
  # 로짓을 0~1 사이 확률값
    rounded_preds = torch.round(torch.sigmoid(preds))
    correct = (rounded_preds == y).float()

    #평균값 계산
    acc = correct.sum() / len(correct)
    return acc

In [ ]:
len(train_loader)

In [ ]:
import torch.optim as optim
optimizer=optim.Adam(model.parameters(), lr=3e-5)

In [ ]:
import time
import torch
import torch.nn as nn
from tqdm.auto import tqdm
from torch.optim import Adam

epochs = 2
optimizer = Adam(model.parameters(), lr=3e-5)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

print(f'Training model with {model_id} on {device}\n')
start_time = time.time()

history = {'loss': [], 'accuracy': [], 'val_loss': [], 'val_accuracy': []}
criterion = nn.BCEWithLogitsLoss()

for epoch in range(epochs):

    model.train()
    train_loss, train_correct = 0, 0

    train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    for batch in train_bar:

        texts, labels = batch


        encoded_input = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        # 라벨 준비
        labels = labels.to(device).float().view(-1, 1)

        # 기울기 초기화
        optimizer.zero_grad()

        # 순전파 (Forward) - encoded_input 안의 ids, mask 등을 풀어 넣음
        logits = model(**encoded_input)
        loss = criterion(logits, labels)

        # 역전파 및 업데이트
        loss.backward()
        optimizer.step()


        # 통계
        train_loss += loss.item()
        preds = torch.sigmoid(logits) >= 0.5
        train_correct += (preds == labels).sum().item()

        train_bar.set_postfix(loss=f"{loss.item():.4f}")


    model.eval()
    val_loss, val_correct = 0, 0

    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            texts, labels = batch

            encoded_input = tokenizer(
                list(texts),
                padding=True,
                truncation=True,
                max_length=128,
                return_tensors="pt"
            ).to(device)

            labels = labels.to(device).float().view(-1, 1)

            logits = model(**encoded_input)
            loss = criterion(logits, labels)

            val_loss += loss.item()
            preds = torch.sigmoid(logits) >= 0.5
            val_correct += (preds == labels).sum().item()


    epoch_train_loss = train_loss / len(train_loader)
    epoch_train_acc = train_correct / len(train_dataset)
    epoch_val_loss = val_loss / len(test_loader)
    epoch_val_acc = val_correct / len(test_dataset)

    history['loss'].append(epoch_train_loss)
    history['accuracy'].append(epoch_train_acc)
    history['val_loss'].append(epoch_val_loss)
    history['val_accuracy'].append(epoch_val_acc)

    print(f">> Epoch {epoch+1} 결과")
    print(f"   Loss: {epoch_train_loss:.4f} | Acc: {epoch_train_acc:.4f} | Val Acc: {epoch_val_acc:.4f}\n")


In [ ]:

model.eval()
test_loss = 0
test_correct = 0

# 기울기 계산 비활성화 (메모리 절약 및 속도 향상)
with torch.no_grad():
    # 진행률 표시 (선택 사항)
    test_bar = tqdm(test_loader, desc="Evaluating")

    for batch in test_bar:
        # 데이터 준비 (문자열 리스트, 라벨 텐서)
        texts, labels = batch

        # 토크나이징 및 GPU 이동
        encoded_input = tokenizer(
            list(texts),
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        labels = labels.to(device).float().view(-1, 1)

        # 모델 예측
        logits = model(**encoded_input)

        # 손실 계산
        loss = criterion(logits, labels)
        test_loss += loss.item()

        # 정확도 계산
        preds = torch.sigmoid(logits) >= 0.5
        test_correct += (preds == labels).sum().item()

# 최종 평균값 계산
final_loss = test_loss / len(test_loader)
final_accuracy = test_correct / len(test_dataset)

print(f'Loss: {final_loss:.4f}')
print(f'Accuracy: {final_accuracy:.4f}')

In [ ]:
import matplotlib.pyplot as plt

# 학습 루프에서 저장한 history 딕셔너리 사용
# print(history.keys()) -> dict_keys(['loss', 'accuracy', 'val_loss', 'val_accuracy'])

acc = history['accuracy']
val_acc = history['val_accuracy']
loss = history['loss']
val_loss = history['val_loss']

epochs_range = range(1, len(acc) + 1)

fig = plt.figure(figsize=(10, 8))

# Loss 그래프
plt.subplot(2, 1, 1)
plt.plot(epochs_range, loss, 'r', label='Training loss')
plt.plot(epochs_range, val_loss, 'b', label='Validation loss')
plt.title('Training and validation loss')
plt.ylabel('Loss')
plt.legend()

# Accuracy 그래프
plt.subplot(2, 1, 2)
plt.plot(epochs_range, acc, 'r', label='Training acc')
plt.plot(epochs_range, val_acc, 'b', label='Validation acc')
plt.title('Training and validation accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

plt.tight_layout() # 서브플롯 간 간격 자동 조절
plt.show()

In [ ]:
import os

# 저장 경로 설정
dataset_name = 'kor_movie'
saved_model_path = './{}_bert'.format(dataset_name.replace('/', '_'))

# 폴더 생성 (경로가 없으면 생성)
os.makedirs(saved_model_path, exist_ok=True)

# 모델 저장 (Hugging Face 방식)

torch.save(model.state_dict(), os.path.join(saved_model_path, 'model_weights.pth'))


# model.bert.save_pretrained(saved_model_path)

print(f"모델이 경로에 저장: {saved_model_path}")

In [ ]:


reloaded_model = BertModel(model_id)

# 저장된 가중치 파일 경로 설정
weights_path = os.path.join(saved_model_path, 'model_weights.pth')

# 가중치 불러오기
state_dict = torch.load(weights_path, map_location=device)
# 딕셔너리 형태 -> 가중치 값들을 모델의 각 레이어에 넣는다
reloaded_model.load_state_dict(state_dict)

reloaded_model.to(device)
reloaded_model.eval()

print(f"모델이 경로에서 성공적으로 로드: {weights_path}")

In [ ]:
def my_review(inputs, results):
    result=[
        f'review:{inputs[i]:<30} : score : {results[i][0]:4f}' for i in range(len(inputs))
    ]

    print(*result, sep='\n')
    print()

examples=[
    text_test[0],
    '영화는 밋밋했다',
    '너무 훌륭한 영화였다',
    '영화는 재미있었다',
    '너무 끔찍하고 잔인한 영화',
    '그럭저럭 재밌었다'
]

reloaded_model.eval()

example_inputs=tokenizer(examples,
                         padding=True,
                         truncation=True,
                         return_tensors="pt").to(device)

with torch.no_grad():
    logits=reloaded_model(**example_inputs)
    reloaded_result=torch.sigmoid(logits).cpu().numpy()


my_review(examples, reloaded_result)
    